# TAE-IA · Module 6 · L17 — Training an Audio Classifier with Transfer Learning

| | |
|---|---|
| **Module** | 6 — Practical AI Applications: Images and Audio |
| **Lesson** | L17 |
| **Track** | B — Audio |
| **Runtime** | **T4 GPU required** — change runtime before running |
| **Drive input** | `ESC50_specs/` — 2000 PNGs from L16 |
| **Drive output** | `ESC50_best.pth` — best model checkpoint (~17 MB) |
| **Drive output** | `ESC50_phase1.pth` — Phase 1 weights, kept for Exercise 1 |

## Learning objectives

By the end of this notebook you will be able to:
1. Load EfficientNet-B0 with ImageNet weights and replace its classifier head for 50 classes
2. Execute two-phase training: frozen backbone → full fine-tune
3. Monitor training with live loss/accuracy curves updated each epoch
4. Save and reload the best checkpoint using `torch.save` / `torch.load`
5. Report final test accuracy on fold 5 and compare with the ESC-50 human baseline (81.3%)

---

## Before running

- **Runtime → Change runtime type → T4 GPU**  
  This notebook will not complete in reasonable time on CPU.
- Verify Drive has ≥5 GB free beyond existing usage.

---

## Cell 0 — Setup

**What the next two cells do:**
- **Setup cell** — mounts Drive, defines the four paths this lab reads and writes
  (`ESC50_DIR`, `SPEC_DIR`, and the two checkpoints), seeds every random number
  generator from `SEED = 42`, and **stops the notebook if there is no GPU**. It also
  counts the PNGs in `ESC50_specs/` and refuses to continue unless all 2000 are there
  — if this fails, go back and finish L16.
- **Imports cell** — installs and imports torch, torchvision, PIL and pandas.

The seeding matters more than it looks: it fixes the validation split, the random
initialisation of the new head, and the shuffle order, so your run is comparable with
your own second run. It does **not** make the result bit-identical — cuDNN picks
convolution algorithms non-deterministically, which is why two people running this
notebook get slightly different numbers.

In [ ]:
# ================================================================
# TAE-IA M6 · Standard Setup -- DO NOT MODIFY THIS CELL
# ================================================================
import os, sys, random, shutil, time
import numpy as np

from google.colab import drive
drive.mount('/content/drive')

MODEL_CACHE = '/content/drive/MyDrive/TAE_IA_M6/models'
os.makedirs(MODEL_CACHE, exist_ok=True)

ESC50_DIR = '/content/drive/MyDrive/TAE_IA_M6/ESC-50'
SPEC_DIR  = '/content/drive/MyDrive/TAE_IA_M6/ESC50_specs'
CKPT_PATH    = '/content/drive/MyDrive/TAE_IA_M6/ESC50_best.pth'
P1_CKPT_PATH = '/content/drive/MyDrive/TAE_IA_M6/ESC50_phase1.pth'   # Phase 1 only — Exercise 1 restarts from this

os.environ['HF_HOME']            = MODEL_CACHE
os.environ['TORCH_HOME']         = MODEL_CACHE
os.environ['TRANSFORMERS_CACHE'] = os.path.join(MODEL_CACHE, 'hub')

SEED = 42
random.seed(SEED); np.random.seed(SEED)

import torch
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

if not torch.cuda.is_available():
    print('\nNo GPU detected. Go to: Runtime > Change runtime type > T4 GPU')
    raise SystemExit('T4 GPU required for L17.')

DEVICE = torch.device('cuda')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# Verify L16 output exists
n_pngs = len([f for f in os.listdir(SPEC_DIR) if f.endswith('.png')]) if os.path.exists(SPEC_DIR) else 0
if n_pngs < 2000:
    raise SystemExit(f'Only {n_pngs}/2000 PNGs found in {SPEC_DIR}. Run L16 first.')
print(f'ESC50_specs: {n_pngs} PNGs found ✓')

In [ ]:
!pip install torchvision pillow tqdm -q

import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
from IPython.display import clear_output

print(f'torch {torch.__version__}  |  torchvision {__import__("torchvision").__version__}')
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})

---
## Section 2.1 — Dataset and DataLoaders

**Key difference from L16:** we resize to **224×224** here (EfficientNet-B0 default) and use **ImageNet normalisation statistics** — not the [0.5, 0.5, 0.5] stats used for generic visualisation.

ImageNet mean/std: `mean=[0.485, 0.456, 0.406]`, `std=[0.229, 0.224, 0.225]`

Using different normalisation than what the backbone was trained on degrades transfer learning quality.

**What the next cell does,** in four steps:
1. **Splits the metadata.** Folds 1–4 (1600 clips) become train+val; fold 5 (400 clips) is
   the test set and is not touched again until Section 2.6. A `sample()` with
   `random_state=SEED` then carves 20% of the 1600 off for validation → **1280 train /
   320 val / 400 test**. Remember from the slides that this validation split is slightly
   leaky (it shuffles at the clip level), which is why the number you *report* is fold 5.
2. **`make_transform(augment)`** — builds the preprocessing chain: resize 128→224,
   optional mild `ColorJitter` for training only, `ToTensor`, repeat the single grey
   channel 3× so the RGB backbone accepts it, then normalise with the **ImageNet**
   mean/std above.
3. **`ESC50Dataset`** — a `torch.utils.data.Dataset`. `__getitem__` maps one metadata row
   to one training example: open `<filename>.png` as greyscale, run it through the
   transform, and return `(image_tensor, target)`.
4. **Three `DataLoader`s** — batch the datasets by 32. Note `shuffle=True` on the
   training loader **only**; `num_workers=2` and `pin_memory=True` overlap disk reads
   with GPU compute.

The last line is the check that has to pass before anything else runs:
**`(32, 3, 224, 224)`**. If the last two numbers are 128, the resize is missing and the
model will train at the wrong resolution without ever raising an error.

In [ ]:
meta = pd.read_csv(os.path.join(ESC50_DIR, 'meta', 'esc50.csv'))

# Split: fold 5 = test, folds 1-4 = train (80% of train used for training, 20% for validation)
train_val_meta = meta[meta['fold'] != 5].reset_index(drop=True)
test_meta      = meta[meta['fold'] == 5].reset_index(drop=True)

# 80/20 split within train+val for a validation set
val_size   = int(0.2 * len(train_val_meta))
val_idx    = train_val_meta.sample(val_size, random_state=SEED).index
train_meta = train_val_meta.drop(val_idx).reset_index(drop=True)
val_meta   = train_val_meta.loc[val_idx].reset_index(drop=True)

print(f'Train: {len(train_meta)}  Val: {len(val_meta)}  Test: {len(test_meta)}')

# ImageNet normalisation — REQUIRED when using ImageNet pretrained weights
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def make_transform(augment=False):
    ops = [T.Resize((224, 224))]
    if augment:
        ops += [T.RandomHorizontalFlip(p=0.0),  # horizontal flip meaningless for spectrograms
                T.ColorJitter(brightness=0.2, contrast=0.2)]  # mild intensity variation
    ops += [
        T.ToTensor(),
        T.Lambda(lambda x: x.repeat(3, 1, 1)),   # grayscale → 3-channel
        T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]
    return T.Compose(ops)


class ESC50Dataset(Dataset):
    def __init__(self, meta_df, spec_dir, transform):
        self.meta      = meta_df
        self.spec_dir  = spec_dir
        self.transform = transform

    def __len__(self):
        return len(self.meta)

    def __getitem__(self, idx):
        row      = self.meta.iloc[idx]
        png_name = row['filename'].replace('.wav', '.png')
        img      = Image.open(os.path.join(self.spec_dir, png_name)).convert('L')
        return self.transform(img), int(row['target'])


BATCH_SIZE = 32

train_ds = ESC50Dataset(train_meta, SPEC_DIR, make_transform(augment=True))
val_ds   = ESC50Dataset(val_meta,   SPEC_DIR, make_transform(augment=False))
test_ds  = ESC50Dataset(test_meta,  SPEC_DIR, make_transform(augment=False))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

# Verify shapes
imgs, labels = next(iter(train_loader))
print(f'Batch: {imgs.shape}  labels: {labels.shape}  range=[{imgs.min():.2f}, {imgs.max():.2f}]')

---
## Section 2.2 — Model Setup

EfficientNet-B0 was trained on ImageNet's 1000 classes. We replace its final `Linear(1280, 1000)` layer with `Linear(1280, 50)` for ESC-50.

**What the next cell does:**
- **`build_model(num_classes, freeze_backbone)`** — the whole lesson in six lines. It
  loads B0 with `IMAGENET1K_V1` weights, *reads* `in_features` off the existing head
  rather than hard-coding 1280 (so the same code works on B2 or ConvNeXt), swaps in a
  fresh `nn.Linear(1280, 50)`, and — if `freeze_backbone=True` — walks
  `model.features.parameters()` setting `requires_grad = False`.
- **`count_params(model)`** — returns `(trainable, total)`. This is how you *prove* the
  freeze happened, rather than assuming it.
- Then it builds the Phase 1 model and pushes a fake `(2, 3, 224, 224)` tensor through it.

Two numbers to check before moving on: **`64,050 / 4,071,598` trainable (1.6%)**, and an
output shape of **`(2, 50)`**. The dummy forward pass is a habit worth keeping — it
separates "the model is wired wrong" from "the data is wrong", which is the most
expensive ambiguity to debug later.

In [ ]:
def build_model(num_classes=50, freeze_backbone=True):
    model = models.efficientnet_b0(
        weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1
    )
    # Replace classifier head
    in_features = model.classifier[1].in_features   # 1280
    model.classifier[1] = nn.Linear(in_features, num_classes)

    if freeze_backbone:
        for param in model.features.parameters():
            param.requires_grad = False

    return model


def count_params(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    return trainable, total


# Phase 1 model — frozen backbone
model = build_model(num_classes=50, freeze_backbone=True).to(DEVICE)
trainable, total = count_params(model)
print(f'Phase 1 — frozen backbone')
print(f'  Trainable params: {trainable:,} / {total:,}  ({trainable/total*100:.1f}%)')
print(f'  Head only:        in_features=1280 → out_features=50')

# Sanity check: forward pass
dummy = torch.randn(2, 3, 224, 224).to(DEVICE)
with torch.no_grad():
    out = model(dummy)
print(f'  Output shape: {out.shape}  (batch=2, classes=50) ✓')

---
## Section 2.3 — Phase 1: Train the Head (Frozen Backbone)

5 epochs, large LR (1e-3), only the 1280→50 linear head trains. This gives the head a good starting point before we unfreeze the backbone.

**The next cell defines three helpers:**
- **`get_lr(optimizer)`** — reads the current learning rate back out. Needed because
  `ReduceLROnPlateau` changes it *silently*: a run that stopped improving because it
  converged and one that stopped because the LR collapsed look identical on the curve.
- **`run_epoch(model, loader, criterion, optimizer=None)`** — one function for both
  training and evaluation; the only difference is whether an optimizer was passed. If it
  was, it calls `model.train()`, enables gradients, and does
  `zero_grad() → backward() → step()`. If not, it calls `model.eval()` and runs under
  `no_grad()`. It returns `(mean_loss, accuracy)`. Note `loss.item() * len(labels)` —
  multiplying by the batch size keeps the mean exact when the last batch is short, and
  `.item()` releases the graph so VRAM does not creep up each epoch.
- **`plot_curves(...)`** — redraws the loss and accuracy panels in place via
  `clear_output(wait=True)`, with the accuracy axis pinned to `(0, 1)` so a 2% wobble
  looks like a 2% wobble.

**Then the cell after that runs Phase 1:** `CrossEntropyLoss` (which applies softmax
internally — do **not** add a softmax layer), an `Adam` optimizer that receives only the
parameters where `requires_grad` is True, and `ReduceLROnPlateau(mode='max')` — `'max'`
because it is watching accuracy, not loss. The loop saves a checkpoint **only when
validation accuracy improves**, and at the end copies the best Phase 1 weights to
`ESC50_phase1.pth`, because Phase 2 is about to overwrite `ESC50_best.pth`.

Watch the first epoch's loss: it should start near **3.9**, which is `ln(50)` — the loss
of a model spreading probability evenly over 50 classes.

In [ ]:
def get_lr(optimizer):
    return optimizer.param_groups[0]['lr']


def run_epoch(model, loader, criterion, optimizer=None):
    """Single train or eval epoch. Pass optimizer=None for eval."""
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss, correct, total = 0.0, 0, 0
    ctx = torch.enable_grad() if is_train else torch.no_grad()

    with ctx:
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            logits = model(imgs)
            loss   = criterion(logits, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * len(labels)
            correct    += (logits.argmax(1) == labels).sum().item()
            total      += len(labels)

    return total_loss / total, correct / total


def plot_curves(train_losses, val_accs, best_val_acc, current_lr, phase_label):
    clear_output(wait=True)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

    epochs = range(1, len(train_losses) + 1)
    ax1.plot(epochs, train_losses, color='#2C75FF', linewidth=2, label='train loss')
    ax1.set_title(f'Training Loss — {phase_label}', fontweight='bold')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Cross-entropy loss')
    ax1.legend()

    ax2.plot(epochs, val_accs, color='#27ae60', linewidth=2, label='val accuracy')
    ax2.axhline(0.813, color='#e67e22', linewidth=1.5, linestyle='--',
                label='human baseline 81.3%')
    ax2.set_title(f'Validation Accuracy — {phase_label}', fontweight='bold')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
    ax2.set_ylim(0, 1.0)
    ax2.legend()

    plt.suptitle(f'Best val acc: {best_val_acc:.3f}   Current LR: {current_lr:.2e}',
                 fontsize=11)
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Phase 1: frozen backbone ──
PHASE1_EPOCHS = 5
LR_PHASE1     = 1e-3

criterion  = nn.CrossEntropyLoss()
optimizer  = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR_PHASE1
)
scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', patience=2, factor=0.5
)

best_val_acc   = 0.0
train_losses_p1 = []
val_accs_p1     = []

print(f'Phase 1: training head only ({PHASE1_EPOCHS} epochs, lr={LR_PHASE1})')
t0 = time.time()

for epoch in range(1, PHASE1_EPOCHS + 1):
    t_loss, t_acc = run_epoch(model, train_loader, criterion, optimizer)
    v_loss, v_acc = run_epoch(model, val_loader,   criterion)

    train_losses_p1.append(t_loss)
    val_accs_p1.append(v_acc)

    if v_acc > best_val_acc:
        best_val_acc = v_acc
        torch.save(model.state_dict(), CKPT_PATH)

    scheduler.step(v_acc)
    plot_curves(train_losses_p1, val_accs_p1, best_val_acc,
                get_lr(optimizer), 'Phase 1 — frozen')
    print(f'  Ep {epoch:2d}/{PHASE1_EPOCHS}  '
          f'train_loss={t_loss:.4f}  train_acc={t_acc:.3f}  '
          f'val_acc={v_acc:.3f}  best={best_val_acc:.3f}')

# Phase 2 overwrites CKPT_PATH, so keep the best Phase 1 weights separately.
# Exercise 1 restarts from this file.
shutil.copy(CKPT_PATH, P1_CKPT_PATH)

print(f'Phase 1 complete in {(time.time()-t0)/60:.1f} min  |  best val acc: {best_val_acc:.3f}')
print(f'Phase 1 checkpoint archived to: {P1_CKPT_PATH}')

---
## Section 2.4 — Phase 2: Full Fine-Tuning

Unfreeze all backbone parameters and continue training with a lower learning rate. The head starts from its Phase 1 weights — the backbone starts from its ImageNet weights and adapts gradually.

**What the next cell does:**
1. **Unfreezes** — loops over `model.features.parameters()` setting `requires_grad = True`.
   The parameter count printed right after should now read **`4,071,598 / 4,071,598`
   (100%)**.
2. **Builds a brand-new optimizer** with *two parameter groups* — the backbone at `1e-4`
   and the head at `1e-3`. This is the **discriminative learning rate** from the slides:
   the backbone holds four million carefully-trained weights that large steps would
   overwrite rather than refine, while the head is still the least-trained part of the
   network. It is a new optimizer on purpose — Adam's momentum estimates were built for
   a different set of trainable parameters and are deliberately discarded.
3. **Runs the training loop** for up to 20 epochs, saving on every improvement and
   stopping early once validation accuracy has not improved for `PATIENCE = 7` epochs.

Watch three things while it runs: the printed **learning rate** (has the scheduler
fired?), the **train/val gap** (is it opening? that is overfitting, and it is expected
here), and the epoch where `best` last changed (how close early stopping is).

In [ ]:
# ── Phase 2: unfreeze all layers ──
PHASE2_EPOCHS = 20
LR_PHASE2     = 1e-4
PATIENCE      = 7

# Unfreeze backbone
for param in model.features.parameters():
    param.requires_grad = True

trainable, total = count_params(model)
print(f'Phase 2 — all layers unfrozen')
print(f'  Trainable params: {trainable:,} / {total:,}  ({trainable/total*100:.1f}%)')

# Differential LR: backbone at LR_PHASE2, head at 10× LR_PHASE2
optimizer = torch.optim.Adam([
    {'params': model.features.parameters(),    'lr': LR_PHASE2},
    {'params': model.classifier.parameters(),  'lr': LR_PHASE2 * 10},
])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', patience=3, factor=0.5
)

train_losses_p2 = []
val_accs_p2     = []
patience_counter = 0

print(f'Phase 2: full fine-tuning (up to {PHASE2_EPOCHS} epochs, '
      f'backbone lr={LR_PHASE2}, head lr={LR_PHASE2*10})')
t0 = time.time()

for epoch in range(1, PHASE2_EPOCHS + 1):
    t_loss, t_acc = run_epoch(model, train_loader, criterion, optimizer)
    v_loss, v_acc = run_epoch(model, val_loader,   criterion)

    train_losses_p2.append(t_loss)
    val_accs_p2.append(v_acc)

    if v_acc > best_val_acc:
        best_val_acc = v_acc
        torch.save(model.state_dict(), CKPT_PATH)
        patience_counter = 0
    else:
        patience_counter += 1

    scheduler.step(v_acc)
    plot_curves(train_losses_p1 + train_losses_p2,
                val_accs_p1 + val_accs_p2,
                best_val_acc, get_lr(optimizer), 'Phase 2 — full fine-tune')
    print(f'  Ep {epoch:2d}/{PHASE2_EPOCHS}  '
          f'train_loss={t_loss:.4f}  train_acc={t_acc:.3f}  '
          f'val_acc={v_acc:.3f}  best={best_val_acc:.3f}  '
          f'patience={patience_counter}/{PATIENCE}')

    if patience_counter >= PATIENCE:
        print(f'Early stopping at epoch {epoch} (patience={PATIENCE})')
        break

print(f'\nPhase 2 complete in {(time.time()-t0)/60:.1f} min  |  best val acc: {best_val_acc:.3f}')
print(f'Best checkpoint saved to: {CKPT_PATH}')

---
## Section 2.5 — Combined Training Curves

Both phases were plotted separately while they ran. This section joins them onto one
pair of axes so the handover is visible.

**What the next cell does:** concatenates the two phases' loss and accuracy lists,
then draws two panels with three reference marks — a vertical dashed line at the
Phase 1→2 boundary, a horizontal line at the 81.3% human baseline, and a horizontal
line at your best validation accuracy.

**Save this figure — Critical Analysis Q1 and Q2 are answered from it.** Look for the
same features the slides described: a small bump or dip in the loss right at the
unfreeze (four million parameters started moving at once), where the accuracy curve
flattens, and whether your best epoch was the last one. It usually is not, which is
exactly why the checkpoint is saved on improvement rather than at the end.

In [ ]:
all_losses = train_losses_p1 + train_losses_p2
all_accs   = val_accs_p1 + val_accs_p2
phase_split = len(train_losses_p1)   # epoch where phase 2 begins

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
epochs = range(1, len(all_losses) + 1)

ax1.plot(epochs, all_losses, color='#2C75FF', linewidth=2)
ax1.axvline(phase_split + 0.5, color='#8e44ad', linewidth=1.5,
            linestyle='--', label='Phase 1→2 (backbone unfrozen)')
ax1.set_title('Training Loss — full run', fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Cross-entropy loss')
ax1.legend()

ax2.plot(epochs, all_accs, color='#27ae60', linewidth=2)
ax2.axvline(phase_split + 0.5, color='#8e44ad', linewidth=1.5,
            linestyle='--', label='Phase 1→2')
ax2.axhline(0.813, color='#e67e22', linewidth=1.5,
            linestyle=':', label='Human baseline 81.3%')
ax2.axhline(best_val_acc, color='#27ae60', linewidth=1,
            linestyle=':', alpha=0.6, label=f'Best val acc {best_val_acc:.3f}')
ax2.set_title('Validation Accuracy — full run', fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.set_ylim(0, 1.0)
ax2.legend()

plt.suptitle(f'EfficientNet-B0 on ESC-50  |  Best val acc: {best_val_acc:.3f}',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Section 2.6 — Test Set Evaluation (Fold 5)

Load the best checkpoint and evaluate on fold 5 — the hold-out set that was never seen during training.

**What the next cell does:** loads `ESC50_best.pth` back into the model with
`load_state_dict`, switches to `model.eval()`, and calls `run_epoch` with no optimizer
so it only measures. It then prints four numbers side by side — your test accuracy,
your best validation accuracy, the 81.3% human baseline and the 2% chance baseline —
and a one-line verdict.

Three details worth knowing, all of which the slides covered:
- **`map_location=DEVICE`** is what lets a checkpoint saved from a GPU load anywhere.
- **`model.eval()` after loading, always.** Forget it and dropout stays active at test
  time and your accuracy drops for no visible reason.
- A `state_dict` is **just a dictionary of tensors** — no architecture. That is why
  `build_model()` has to be called first, and why L18 can reload this file.

**Expect validation accuracy to come out above test accuracy.** That is not a bug: part
of it is overfitting and part is the leak in the validation split. The number you quote
to anyone is the fold-5 one. **Write it down** — L18 opens by comparing against it.

In [ ]:
# Load best checkpoint
model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
model.eval()
print(f'Loaded checkpoint: {CKPT_PATH}')

_, test_acc = run_epoch(model, test_loader, criterion)

print()
print('=' * 50)
print(f'  Test accuracy (fold 5): {test_acc:.4f}  ({test_acc*100:.1f}%)')
print(f'  Best val accuracy:      {best_val_acc:.4f}  ({best_val_acc*100:.1f}%)')
print(f'  Human baseline:          0.813  (81.3%)')
print(f'  Chance baseline:        ~0.020  (~2.0%)')
print('=' * 50)

gap_from_human = 0.813 - test_acc
if gap_from_human <= 0:
    print(f'  Result: matched or exceeded human baseline!')
elif gap_from_human <= 0.05:
    print(f'  Result: {gap_from_human*100:.1f}% below human — strong result')
elif gap_from_human <= 0.15:
    print(f'  Result: {gap_from_human*100:.1f}% below human — good for 1600 training clips')
else:
    print(f'  Result: {gap_from_human*100:.1f}% below human — check training curves for issues')

---
## Section 2.7 — Per-Class Accuracy

Global accuracy hides which categories are easy and which are hard. We compute per-class accuracy here as a preview of L18's full confusion matrix analysis.

**What the next cell does:** runs the model over the whole test loader once more,
collecting every prediction and every true label into two arrays. It then computes, for
each of the 50 classes, the fraction of that class's clips that were classified
correctly, and draws them as a **sorted horizontal bar chart** — red below 50%, orange
in between, green at or above 75% — with a dashed line at your overall accuracy. Finally
it prints the five easiest and five hardest categories by name.

**Read the chart as a ranking, not a measurement.** Fold 5 has **eight clips per class**,
so every bar can only take one of nine values (0, 1/8, 2/8 … 1). A class at 0.50 got four
of eight right, and a different eight clips could have given three or five without
anything having changed. L18 puts proper confidence intervals on this.

Note your **three worst categories by name** — Critical Analysis Q3 asks for one concrete
fix for each, and L18 starts from them.

In [ ]:
# Collect all predictions
all_preds, all_targets = [], []
model.eval()
with torch.no_grad():
    for imgs, labels in test_loader:
        preds = model(imgs.to(DEVICE)).argmax(1).cpu()
        all_preds.append(preds)
        all_targets.append(labels)

all_preds   = torch.cat(all_preds).numpy()
all_targets = torch.cat(all_targets).numpy()

# Per-class accuracy
category_map = meta.drop_duplicates('target').set_index('target')['category'].to_dict()
per_class = {}
for cls_id in range(50):
    mask = all_targets == cls_id
    if mask.sum() > 0:
        per_class[cls_id] = (all_preds[mask] == cls_id).mean()

sorted_cls = sorted(per_class.items(), key=lambda x: x[1])
cat_names  = [category_map[c] for c, _ in sorted_cls]
accs       = [a for _, a in sorted_cls]

fig, ax = plt.subplots(figsize=(14, 10))
colors  = ['#c0392b' if a < 0.5 else '#27ae60' if a >= 0.75 else '#e67e22' for a in accs]
ax.barh(cat_names, accs, color=colors, edgecolor='white', linewidth=0.4)
ax.axvline(test_acc, color='#2C75FF', linewidth=1.5, linestyle='--',
           label=f'Overall test acc: {test_acc:.2f}')
ax.set_xlabel('Per-class accuracy')
ax.set_title('Per-class accuracy on test fold (fold 5)', fontweight='bold')
ax.set_xlim(0, 1.05)
ax.legend()
plt.tight_layout()
plt.show()

print('\nTop 5 easiest categories:')
for cls_id, acc in sorted_cls[-5:][::-1]:
    print(f'  {category_map[cls_id]:<22}: {acc:.2f}')
print('\nTop 5 hardest categories:')
for cls_id, acc in sorted_cls[:5]:
    print(f'  {category_map[cls_id]:<22}: {acc:.2f}')

---
## Exercise 1 — Learning rate sensitivity

Phase 2 used a differential LR: `backbone=1e-4, head=1e-3`. Re-run phase 2 with a **uniform LR of 1e-4** (same for both backbone and head) and compare the validation accuracy curves.

1. Plot both curves on the same axes
2. Which converges faster? Which reaches higher accuracy?
3. In a markdown cell: explain why differential LR helps or doesn't help in this setting

> Tip: restart from `P1_CKPT_PATH` — the Phase 1 weights archived at the end of Section 2.3.
> Do **not** use `CKPT_PATH`: Phase 2 overwrote it with the fine-tuned model, so continuing
> from it would compare 20 epochs against 40 and tell you nothing.

In [ ]:
# TODO: re-run phase 2 with uniform LR=1e-4 and compare
# Start from the archived PHASE 1 weights, not CKPT_PATH:
# model_uniform = build_model(num_classes=50, freeze_backbone=False).to(DEVICE)
# model_uniform.load_state_dict(torch.load(P1_CKPT_PATH, map_location=DEVICE))
# optimizer_uniform = torch.optim.Adam(model_uniform.parameters(), lr=1e-4)
# ... run the same training loop, collect val_accs_uniform ...
# ... then plot val_accs_p2 and val_accs_uniform on one axis ...

**Exercise 1 — Answer:**

[YOUR ANSWER]

---
## Exercise 2 — Batch size effect on training stability

The notebook uses `batch_size=32`. Without retraining, answer the following from theory:

1. If you doubled the batch size to 64 with the same LR, what would happen to the gradient estimates? Would training be faster or slower per epoch?
2. If you halved it to 16, what tradeoff would you face?
3. Given that EfficientNet-B0 uses ~3 GB VRAM at batch 32 — of which the weights and Adam state are only ~65 MB, and ~1 GB is CUDA context and fragmentation — estimate whether batch 64 would fit on a T4 (15.6 GB total). Which term actually scales with batch size?

Answer in the markdown cell below.

**Exercise 2 — Answer:**

[YOUR ANSWER — ~4 sentences covering gradient noise, training speed, and VRAM estimate]

---
## Part 4 — Critical Analysis

### Q1 — Two-phase training rationale

From your training curves: compare the validation accuracy at the end of Phase 1 vs. the end of Phase 2. Explain in 3 sentences why Phase 1 (frozen backbone) is necessary before Phase 2. What would likely happen if you started Phase 2 directly with a randomly-initialised head?

**[YOUR ANSWER]**

---

### Q2 — Train/val gap interpretation

From your final training curves: is there a gap between train accuracy and val accuracy? If yes, what does this indicate? What is the standard remedy, and why is it particularly important with only 1,280 training clips?

**[YOUR ANSWER]** *(~3 sentences)*

---

### Q3 — Hard categories

From Section 2.7: list the 3 hardest categories your model struggled with. For each one, propose **one specific change** to the preprocessing or training pipeline that could improve accuracy on that category. Be concrete — not just "more data" or "more epochs".

**[YOUR ANSWER]**

---

### Q4 — Deployment considerations

Your model achieved X% accuracy on ESC-50 fold 5. A company wants to deploy it in a smart-home device to detect specific sounds (e.g., breaking glass, smoke alarm, dog bark). Name **two reasons** why the fold-5 accuracy is an optimistic estimate of real-world performance, and one concrete step you would take before deployment.

**[YOUR ANSWER]** *(~4 sentences)*

---

---
## Submission Checklist

Before saving and submitting:

- [ ] T4 GPU confirmed in Cell 0
- [ ] ESC50_specs: 2000 PNGs verified
- [ ] DataLoader batch shape `(32, 3, 224, 224)` confirmed
- [ ] Phase 1 completed — loss/accuracy curves plotted
- [ ] Phase 2 completed — best checkpoint saved to `ESC50_best.pth`
- [ ] Combined curves plotted with phase boundary marker
- [ ] Test accuracy on fold 5 printed and compared with human baseline
- [ ] Per-class accuracy bar chart plotted — top-5 easy and hard categories identified
- [ ] Exercise 1 attempted (LR sensitivity — even a written answer is acceptable)
- [ ] Exercise 2 answered in markdown
- [ ] Critical Analysis Q1–Q4 answered
- [ ] Notebook saved to Drive

**For L18:** `ESC50_best.pth` must be on Drive. Record your test accuracy here: **_____%**

---
*TAE-IA 2025 · Cocyten-Nayarit · Module 6 · L17*  
*Track B — Audio | Next: L18 — Evaluating and Diagnosing the Classifier*